# TechStore Plus — Initial Customer Service Chatbot Project

This notebook implements a customer service chatbot for **TechStore Plus**, including:

1. Professional customer service assistant role
2. Conversational context handling
3. Customer information collection
4. Inquiry type identification and routing
5. Query sentiment, emotion, category, urgency, and entity extraction
6. Personalized response generation
7. Conversation persistence in JSON files
8. Consolidation of multiple conversations into one JSON file
9. Usage examples and test cases

> This project was designed to be easy to explain in a video or technical presentation.

## 1. Project Setup

This section imports the required libraries and configures the environment.

Prompting techniques used in this notebook:
- **System role prompting**: defines the assistant behavior.
- **Structured output prompting**: requests JSON responses from the model.
- **Context-aware prompting**: passes conversation history to maintain continuity.
- **Few-shot/testing prompts**: validates the chatbot with sample queries.

In [1]:
# Standard Python libraries used for data handling, timestamps, files, and JSON persistence.
import os
import json
import uuid
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional

# OpenAI client.
# Install if needed:
# !pip install openai python-dotenv

from openai import OpenAI

# Optional: load API key from a .env file if you have one.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# The OpenAI client reads OPENAI_API_KEY from your environment.
client = OpenAI()

# Folder where all generated JSON conversation files will be stored.
DATA_DIR = Path("conversation_data")
DATA_DIR.mkdir(exist_ok=True)

## 2. Fictional Company Context

The chatbot needs company-specific context so it can answer consistently and route customers properly.

In [2]:
# Fictional company knowledge base used by the assistant.
COMPANY_CONTEXT = {
    "company_name": "TechStore Plus",
    "description": "Your Trusted Technology Store",
    "sector": "E-commerce for technology products",
    "founded": 2018,
    "location": "New York, USA",
    "format": "Online store with physical showroom",
    "business_hours": {
        "monday_friday": "09:00-18:00",
        "saturday": "10:00-14:00"
    },
    "products": [
        "Laptops",
        "Desktop PCs",
        "Workstations",
        "Smartphones",
        "Tablets",
        "Headphones",
        "Mice",
        "Keyboards",
        "Cables",
        "Cases",
        "Gaming consoles",
        "Games",
        "Gaming accessories",
        "IoT devices",
        "Smart speakers",
        "Cameras"
    ],
    "services": [
        "Nationwide shipping",
        "Technical support",
        "Installation",
        "Configuration",
        "Diagnosis",
        "Warranty",
        "Financing",
        "Trade-in"
    ],
    "policies": {
        "shipping": "Free nationwide shipping for purchases over $500.",
        "returns": "30 days for exchanges and 7 days for refunds.",
        "warranty": "12 months warranty on all products.",
        "installation": "Home technical service is available.",
        "extended_warranty": "Optional extended warranty for 1 additional year.",
        "financing": "Interest-free installments and payment plans are available."
    },
    "contact": {
        "email": "support@techstoreplus.com",
        "phone": "1-800-TECH-PLUS",
        "chat": "Available on website 24/7",
        "text": "+1 555-123-4567"
    }
}

## 3. System Role and Chatbot Configuration

This prompt defines the assistant as a professional customer service agent.
It also instructs the assistant to collect customer information, identify the issue, route the case, and keep a natural conversation.

In [3]:
SYSTEM_ROLE = f"""
You are a professional customer service assistant for {COMPANY_CONTEXT['company_name']}.

Your behavior:
- Be polite, helpful, clear, and professional.
- Maintain a natural conversation.
- Adapt your tone to the customer's tone.
- If the customer is upset or urgent, acknowledge the situation with empathy.
- Ask for missing information only when necessary.
- Provide clear next steps.
- Do not invent unavailable company policies.
- Use the company context below as your source of truth.

Company context:
{json.dumps(COMPANY_CONTEXT, indent=2)}

Conversation flow:
1. Personalized greeting.
2. Collect basic customer information when needed:
   - customer name
   - email or phone
   - order number if applicable
3. Identify inquiry type:
   - technical
   - billing
   - return
   - warranty
   - product_information
   - installation
   - financing
   - general_information
4. Route the case:
   - Technical issue: technical support team
   - Billing/receipt/payment: billing department
   - Return/refund/exchange: returns department
   - Warranty: warranty support team
   - Product recommendation: sales advisor
   - Installation request: installation scheduling team
5. Give specific and practical next steps.
"""

## 4. Conversation Context Handling

The following class stores conversation history and keeps the customer interaction contextual.

In [4]:
class ConversationSession:
    """
    Stores the conversation history for one customer session.

    This supports conversational context handling by sending previous messages
    back to the model on each request.
    """

    def __init__(self, customer_id: Optional[str] = None):
        self.customer_id = customer_id or f"CUST-{uuid.uuid4().hex[:8].upper()}"
        self.created_at = datetime.now().isoformat(timespec="seconds")
        self.messages: List[Dict[str, str]] = [
            {"role": "system", "content": SYSTEM_ROLE}
        ]

    def add_user_message(self, message: str):
        self.messages.append({"role": "user", "content": message})

    def add_assistant_message(self, message: str):
        self.messages.append({"role": "assistant", "content": message})

    def get_public_history(self) -> List[Dict[str, str]]:
        """Returns conversation messages without the system role."""
        return [m for m in self.messages if m["role"] != "system"]

## 5. Query Analysis and Classification

This function analyzes a customer query and returns structured JSON with:
- sentiment
- emotions
- category
- urgency
- extracted entities
- routing recommendation

In [5]:
def analyze_customer_query(query: str) -> Dict[str, Any]:
    """
    Uses structured output prompting to classify and analyze the customer's query.
    """

    prompt = f"""
Analyze the following customer service query.

Return ONLY valid JSON with this exact structure:
{{
  "customer_sentiment": "positive | neutral | negative",
  "detected_emotions": ["frustration", "urgency", "satisfaction", "confusion", "concern"],
  "query_category": "technical | billing | return | warranty | product_information | installation | financing | general_information",
  "urgency_level": "low | medium | high",
  "mentioned_products": ["product names if any"],
  "extracted_information": {{
    "order_number": "if_applicable",
    "purchase_date": "if_applicable",
    "amount": "if_applicable",
    "location": "if_applicable",
    "budget": "if_applicable"
  }},
  "recommended_routing": "team or department name",
  "reasoning_summary": "short explanation"
}}

Customer query:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an expert customer service query classifier. Always return valid JSON only."},
            {"role": "user", "content": prompt}
        ]
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # Basic fallback in case the model returns extra formatting.
        cleaned = content.strip().replace("```json", "").replace("```", "").strip()
        return json.loads(cleaned)

## 6. Personalized Response Generation

This function generates a response adapted to:
- customer tone
- sentiment
- urgency
- query category
- extracted information
- company policies

In [6]:
def generate_personalized_response(
    session: ConversationSession,
    user_query: str,
    analysis: Dict[str, Any]
) -> str:
    """
    Generates a customer service response using:
    - conversation context
    - customer query analysis
    - TechStore Plus policies
    """

    response_instruction = f"""
Customer query analysis:
{json.dumps(analysis, indent=2)}

Generate a professional customer service response.

Requirements:
- Match the customer's tone.
- If sentiment is negative, start with empathy.
- If urgency is high, acknowledge priority and provide immediate next steps.
- Personalize the response according to the query category.
- Include specific information from the company policy when relevant.
- Ask for missing information if needed.
- Keep the answer clear and practical.
- End with the next best action.
"""

    # Add the user query to the conversation.
    session.add_user_message(user_query)

    # Send conversation history + the response instruction.
    messages = session.messages + [
        {"role": "user", "content": response_instruction}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.4,
        messages=messages
    )

    assistant_response = response.choices[0].message.content
    session.add_assistant_message(assistant_response)

    return assistant_response

## 7. Main Chatbot Function

This function combines:
1. query analysis
2. response generation
3. conversation context update

In [7]:
def chatbot_reply(session: ConversationSession, user_query: str) -> Dict[str, Any]:
    """
    Main function to process a customer message.
    """

    analysis = analyze_customer_query(user_query)
    reply = generate_personalized_response(session, user_query, analysis)

    return {
        "customer_id": session.customer_id,
        "query": user_query,
        "analysis": analysis,
        "reply": reply
    }

## 8. Conversation Summary and JSON Persistence

This section creates structured summaries and saves each conversation into an individual JSON file with a timestamp.

In [8]:
def generate_conversation_summary(
    session: ConversationSession,
    latest_analysis: Dict[str, Any],
    resolution_status: str = "pending",
    actions_taken: Optional[List[str]] = None,
    follow_up_required: bool = True
) -> Dict[str, Any]:
    """
    Creates a structured JSON summary using the required format.
    """

    actions_taken = actions_taken or []

    public_history = session.get_public_history()

    summary_prompt = f"""
Summarize the following customer service conversation in one concise paragraph.

Conversation:
{json.dumps(public_history, indent=2)}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "system", "content": "You summarize customer service conversations clearly and concisely."},
            {"role": "user", "content": summary_prompt}
        ]
    )

    concise_summary = response.choices[0].message.content

    conversation_json = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "customer_id": session.customer_id,
        "conversation_summary": concise_summary,
        "query_category": latest_analysis.get("query_category", "general_information"),
        "customer_sentiment": latest_analysis.get("customer_sentiment", "neutral"),
        "urgency_level": latest_analysis.get("urgency_level", "low"),
        "mentioned_products": latest_analysis.get("mentioned_products", []),
        "extracted_information": latest_analysis.get("extracted_information", {}),
        "resolution_status": resolution_status,
        "actions_taken": actions_taken,
        "follow_up_required": follow_up_required
    }

    return conversation_json


def save_conversation_json(conversation_summary: Dict[str, Any]) -> Path:
    """
    Saves each conversation summary in an individual JSON file with timestamp.
    """

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    customer_id = conversation_summary["customer_id"]
    file_path = DATA_DIR / f"conversation_{customer_id}_{timestamp}.json"

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(conversation_summary, file, indent=2, ensure_ascii=False)

    return file_path

## 9. Consolidating Multiple Conversations

This function reads all individual conversation JSON files and compiles them into one consolidated file.

In [9]:
def consolidate_conversations(output_file: str = "consolidated_conversations.json") -> Path:
    """
    Compiles multiple individual conversation JSON files into one consolidated JSON file.
    """

    all_conversations = []

    for file_path in DATA_DIR.glob("conversation_*.json"):
        with open(file_path, "r", encoding="utf-8") as file:
            all_conversations.append(json.load(file))

    consolidated_data = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "total_conversations": len(all_conversations),
        "conversations": all_conversations
    }

    output_path = DATA_DIR / output_file

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(consolidated_data, file, indent=2, ensure_ascii=False)

    return output_path

In [13]:
# ============================================================
# MOCK MODE - No OpenAI API calls, no token usage
# ============================================================
# This cell overrides the OpenAI-based functions with local
# rule-based functions. It allows the project to be demonstrated
# without consuming API credits or tokens.

MOCK_MODE = True

import re
import json
from datetime import datetime
from typing import Dict, List, Any, Optional


def mock_extract_products(query: str) -> List[str]:
    """
    Extracts known product names from the customer query using simple rules.
    This replaces AI-based entity extraction in mock mode.
    """

    product_keywords = {
        "iphone 15": "iPhone 15",
        "iphone": "iPhone",
        "laptop": "Laptop",
        "gaming headphones": "Gaming Headphones",
        "headphones": "Headphones",
        "router": "Router",
        "tablet": "Tablet",
        "macbook pro": "MacBook Pro",
        "phone": "Phone",
        "home theater": "Home Theater"
    }

    lower_query = query.lower()
    products = []

    for keyword, product_name in product_keywords.items():
        if keyword in lower_query and product_name not in products:
            products.append(product_name)

    return products


def mock_extract_information(query: str) -> Dict[str, Any]:
    """
    Extracts structured information such as order number, dates and amounts.
    """

    order_match = re.search(r"#?[A-Z]{3}-\d{4}-\d{3}", query)
    amount_match = re.search(r"\$\s?\d+(?:\.\d{2})?", query)
    date_match = re.search(
        r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}(?:st|nd|rd|th)?",
        query,
        re.IGNORECASE
    )

    return {
        "order_number": order_match.group(0) if order_match else None,
        "purchase_date": date_match.group(0) if date_match else None,
        "amount": amount_match.group(0) if amount_match else None
    }


def analyze_customer_query(query: str) -> Dict[str, Any]:
    """
    MOCK VERSION of analyze_customer_query.

    This function does not call OpenAI.
    It uses local Python rules to classify:
    - sentiment
    - emotions
    - category
    - urgency
    - products
    - entities
    - routing
    """

    lower = query.lower()

    # Urgency detection
    if any(word in lower for word in ["emergency", "urgent", "urgently", "tomorrow", "asap", "immediately"]):
        urgency = "high"
    elif any(word in lower for word in ["can't", "doesn't work", "problem", "warranty", "receipt", "need"]):
        urgency = "medium"
    else:
        urgency = "low"

    # Sentiment analysis
    if any(word in lower for word in ["thank", "excellent", "great", "happy", "satisfied"]):
        sentiment = "positive"
    elif any(word in lower for word in ["emergency", "never arrived", "can't", "doesn't work", "frustrated", "not compatible"]):
        sentiment = "negative"
    else:
        sentiment = "neutral"

    # Emotion identification
    emotions = []

    if sentiment == "positive":
        emotions.append("satisfaction")

    if any(word in lower for word in ["emergency", "urgent", "urgently", "tomorrow"]):
        emotions.append("urgency")

    if any(word in lower for word in ["can't", "doesn't work", "tried everything", "never arrived", "frustrated"]):
        emotions.append("frustration")

    if any(word in lower for word in ["how", "what", "recommend", "do you have"]):
        emotions.append("curiosity")

    if not emotions:
        emotions.append("neutral")

    # Category classification
    if any(word in lower for word in ["receipt", "billing", "payment", "visa", "installments", "financing"]):
        if any(word in lower for word in ["installments", "financing", "visa"]):
            category = "financing"
        else:
            category = "billing"

    elif any(word in lower for word in ["return", "refund", "exchange"]):
        category = "return"

    elif "warranty" in lower or "won't turn on" in lower:
        category = "warranty"

    elif any(word in lower for word in ["configure", "install", "installation", "doesn't work", "technical"]):
        if "install" in lower or "installation" in lower:
            category = "installation"
        else:
            category = "technical"

    elif any(word in lower for word in ["stock", "shipping", "recommend", "budget", "how much"]):
        category = "product_information"

    else:
        category = "general_information"

    # Routing decision
    routing_map = {
        "technical": "Technical Support Team",
        "billing": "Billing Team",
        "return": "Returns Team",
        "warranty": "Warranty Support Team",
        "product_information": "Sales/Product Information Team",
        "installation": "Installation Services Team",
        "financing": "Billing and Financing Team",
        "general_information": "General Customer Service Team"
    }

    if urgency == "high":
        recommended_routing = "Priority Support Team"
    else:
        recommended_routing = routing_map.get(category, "General Customer Service Team")

    return {
        "customer_sentiment": sentiment,
        "detected_emotions": emotions,
        "query_category": category,
        "urgency_level": urgency,
        "mentioned_products": mock_extract_products(query),
        "extracted_information": mock_extract_information(query),
        "recommended_routing": recommended_routing,
        "reasoning_summary": "Mock rule-based analysis generated locally without OpenAI API calls."
    }


def generate_personalized_response(
    session: ConversationSession,
    user_query: str,
    analysis: Dict[str, Any]
) -> str:
    """
    MOCK VERSION of generate_personalized_response.

    This function generates a customer service response locally.
    It does not call OpenAI.
    """

    sentiment = analysis.get("customer_sentiment", "neutral")
    urgency = analysis.get("urgency_level", "low")
    category = analysis.get("query_category", "general_information")
    routing = analysis.get("recommended_routing", "Customer Service Team")
    products = analysis.get("mentioned_products", [])
    extracted = analysis.get("extracted_information", {})
    order_number = extracted.get("order_number")

    # Opening tone
    if urgency == "high":
        opening = "I understand this is urgent, and I’m sorry for the inconvenience."
    elif sentiment == "positive":
        opening = "Thank you for your kind feedback. I’m glad to help you with your request."
    elif sentiment == "negative":
        opening = "I’m sorry you’re experiencing this issue. I’ll help you with the next steps."
    else:
        opening = "Thank you for contacting TechStore Plus. I’ll be happy to assist you."

    # Category-specific response
    if category == "technical":
        action = "I will route your case to our Technical Support Team for troubleshooting and configuration assistance."

    elif category == "billing":
        action = "I will route your request to our Billing Team so they can help with your receipt or payment details."

    elif category == "return":
        action = "I can help you start the return process. TechStore Plus offers 30 days for exchanges and 7 days for refunds."

    elif category == "warranty":
        action = "I can help you with the warranty process. TechStore Plus provides 12 months of warranty on all products."

    elif category == "installation":
        action = "I can route your request to our Installation Services Team. Home technical service is available."

    elif category == "financing":
        action = "I can help with financing information. TechStore Plus offers interest-free installments and payment plans."

    elif category == "product_information":
        action = "I can help with product availability, recommendations, shipping details and purchase options."

    else:
        action = "I can help with general information about products, services, policies and support channels."

    # Entity-specific detail
    details = []

    if order_number:
        details.append(f"I found your order number: {order_number}.")

    if products:
        details.append(f"Product mentioned: {', '.join(products)}.")

    details.append(f"Recommended routing: {routing}.")

    if urgency == "high":
        details.append("Because this case is urgent, it should be prioritized for faster handling.")

    reply = f"{opening} {action} {' '.join(details)}"

    # Important: update conversation context locally
    session.add_user_message(user_query)
    session.add_assistant_message(reply)

    return reply


def generate_conversation_summary(
    session: ConversationSession,
    latest_analysis: Dict[str, Any],
    resolution_status: str = "pending",
    actions_taken: Optional[List[str]] = None,
    follow_up_required: bool = True
) -> Dict[str, Any]:
    """
    MOCK VERSION of generate_conversation_summary.

    This creates the required JSON structure without calling OpenAI.
    """

    actions_taken = actions_taken or []
    public_history = session.get_public_history()

    # Build a concise summary locally
    customer_messages = [m["content"] for m in public_history if m["role"] == "user"]
    assistant_messages = [m["content"] for m in public_history if m["role"] == "assistant"]

    if customer_messages:
        main_customer_request = customer_messages[-1]
    else:
        main_customer_request = "No customer message available."

    concise_summary = (
        f"Customer contacted TechStore Plus about a {latest_analysis.get('query_category', 'general_information')} issue. "
        f"The detected sentiment was {latest_analysis.get('customer_sentiment', 'neutral')} and urgency was "
        f"{latest_analysis.get('urgency_level', 'low')}. "
        f"Main request: {main_customer_request}"
    )

    conversation_json = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "customer_id": session.customer_id,
        "conversation_summary": concise_summary,
        "query_category": latest_analysis.get("query_category", "general_information"),
        "customer_sentiment": latest_analysis.get("customer_sentiment", "neutral"),
        "urgency_level": latest_analysis.get("urgency_level", "low"),
        "mentioned_products": latest_analysis.get("mentioned_products", []),
        "extracted_information": latest_analysis.get("extracted_information", {}),
        "resolution_status": resolution_status,
        "actions_taken": actions_taken,
        "follow_up_required": follow_up_required
    }

    return conversation_json


print("MOCK MODE ACTIVE: OpenAI API calls are disabled for analysis, response generation and summaries.")

MOCK MODE ACTIVE: OpenAI API calls are disabled for analysis, response generation and summaries.


## 10. Usage Example

This example shows how to create a session, receive a customer query, analyze it, generate a response, and save the conversation.

In [14]:
# Example customer session.
session = ConversationSession()

# Example customer query.
user_query = "This is an emergency! My order #TEC-2024-001 never arrived and I need that laptop for work tomorrow!"

# Process the customer message.
result = chatbot_reply(session, user_query)

print("CUSTOMER ID:")
print(result["customer_id"])

print("\nQUERY ANALYSIS:")
print(json.dumps(result["analysis"], indent=2))

print("\nCHATBOT RESPONSE:")
print(result["reply"])

# Create and save the required JSON summary.
summary = generate_conversation_summary(
    session=session,
    latest_analysis=result["analysis"],
    resolution_status="escalated",
    actions_taken=[
        "Acknowledged urgent delivery issue",
        "Requested confirmation of delivery address and contact information",
        "Routed case to priority support"
    ],
    follow_up_required=True
)

saved_file = save_conversation_json(summary)
print(f"\nConversation saved to: {saved_file}")

CUSTOMER ID:
CUST-78C546C3

QUERY ANALYSIS:
{
  "customer_sentiment": "negative",
  "detected_emotions": [
    "urgency",
    "frustration"
  ],
  "query_category": "general_information",
  "urgency_level": "high",
  "mentioned_products": [
    "Laptop"
  ],
  "extracted_information": {
    "order_number": "#TEC-2024-001",
    "purchase_date": null,
    "amount": null
  },
  "recommended_routing": "Priority Support Team",
  "reasoning_summary": "Mock rule-based analysis generated locally without OpenAI API calls."
}

CHATBOT RESPONSE:
I understand this is urgent, and I’m sorry for the inconvenience. I can help with general information about products, services, policies and support channels. I found your order number: #TEC-2024-001. Product mentioned: Laptop. Recommended routing: Priority Support Team. Because this case is urgent, it should be prioritized for faster handling.

Conversation saved to: conversation_data\conversation_CUST-78C546C3_20260514_185314.json


## 11. Test Cases

The following test cases are based on the suggested customer queries.
They validate different categories, tones, and urgency levels.

In [15]:
TEST_QUERIES = [
    "Hello, I’d like to know if you have the new iPhone 15 in stock and how much shipping costs to Chicago",
    "This is an emergency! My order #TEC-2024-001 never arrived and I need that laptop for work tomorrow!",
    "Thank you so much for the excellent service with my previous purchase, I want to buy gaming headphones",
    "I can’t configure the router I bought last week, I’ve tried everything and it doesn’t work",
    "Good morning, I need the receipt for my purchase from December 15th, order #TEC-2023-089",
    "I bought a tablet 8 months ago and now it won’t turn on, how do I use the warranty?",
    "What laptop do you recommend for an engineering student? Maximum budget $800",
    "I need to urgently return this phone I bought yesterday, it’s not compatible with my plan",
    "Can you come install the home theater I bought? I live in downtown",
    "Do you have interest-free installments for the MacBook Pro? Do you accept Visa cards?"
]

In [16]:
# Run all test cases.
test_results = []

for query in TEST_QUERIES:
    test_session = ConversationSession()
    result = chatbot_reply(test_session, query)

    test_results.append({
        "query": query,
        "customer_id": result["customer_id"],
        "category": result["analysis"].get("query_category"),
        "sentiment": result["analysis"].get("customer_sentiment"),
        "urgency": result["analysis"].get("urgency_level"),
        "routing": result["analysis"].get("recommended_routing"),
        "reply": result["reply"]
    })

# Display compact results.
for item in test_results:
    print("=" * 100)
    print(f"QUERY: {item['query']}")
    print(f"CATEGORY: {item['category']}")
    print(f"SENTIMENT: {item['sentiment']}")
    print(f"URGENCY: {item['urgency']}")
    print(f"ROUTING: {item['routing']}")
    print(f"REPLY: {item['reply']}")
    print()

QUERY: Hello, I’d like to know if you have the new iPhone 15 in stock and how much shipping costs to Chicago
CATEGORY: product_information
SENTIMENT: neutral
URGENCY: low
ROUTING: Sales/Product Information Team
REPLY: Thank you for contacting TechStore Plus. I’ll be happy to assist you. I can help with product availability, recommendations, shipping details and purchase options. Product mentioned: iPhone 15, iPhone, Phone. Recommended routing: Sales/Product Information Team.

QUERY: This is an emergency! My order #TEC-2024-001 never arrived and I need that laptop for work tomorrow!
CATEGORY: general_information
SENTIMENT: negative
URGENCY: high
ROUTING: Priority Support Team
REPLY: I understand this is urgent, and I’m sorry for the inconvenience. I can help with general information about products, services, policies and support channels. I found your order number: #TEC-2024-001. Product mentioned: Laptop. Recommended routing: Priority Support Team. Because this case is urgent, it shoul

## 12. Saving Test Case Summaries

This section saves one JSON file for each tested conversation and then creates a consolidated JSON file.

In [17]:
# Save summaries for all test cases.
saved_files = []

for query in TEST_QUERIES:
    test_session = ConversationSession()
    result = chatbot_reply(test_session, query)

    # Simple status rule for demonstration.
    urgency = result["analysis"].get("urgency_level", "low")
    resolution_status = "escalated" if urgency == "high" else "pending"

    summary = generate_conversation_summary(
        session=test_session,
        latest_analysis=result["analysis"],
        resolution_status=resolution_status,
        actions_taken=[
            f"Classified query as {result['analysis'].get('query_category')}",
            f"Routed to {result['analysis'].get('recommended_routing')}"
        ],
        follow_up_required=True if resolution_status in ["pending", "escalated"] else False
    )

    saved_files.append(save_conversation_json(summary))

print("Saved files:")
for file in saved_files:
    print(file)

consolidated_file = consolidate_conversations()
print(f"\nConsolidated file created at: {consolidated_file}")

Saved files:
conversation_data\conversation_CUST-43C06F94_20260514_193951.json
conversation_data\conversation_CUST-14EC7816_20260514_193951.json
conversation_data\conversation_CUST-7DB12C0B_20260514_193951.json
conversation_data\conversation_CUST-B872858F_20260514_193951.json
conversation_data\conversation_CUST-F2E23AC2_20260514_193951.json
conversation_data\conversation_CUST-299B1FA5_20260514_193951.json
conversation_data\conversation_CUST-8696A2D2_20260514_193951.json
conversation_data\conversation_CUST-CA926D5F_20260514_193951.json
conversation_data\conversation_CUST-E49B70EA_20260514_193951.json
conversation_data\conversation_CUST-786E0A75_20260514_193951.json

Consolidated file created at: conversation_data\consolidated_conversations.json


## 13. Results Analysis

After running the notebook, check the following:

- Whether urgent cases are classified as `high`.
- Whether negative messages receive empathetic responses.
- Whether billing, warranty, return, technical, installation, financing, and product queries are routed correctly.
- Whether extracted entities such as order numbers, dates, products, locations, and budgets are captured.
- Whether each conversation summary follows the required JSON structure.
- Whether the consolidated JSON file includes all individual conversations.

Possible improvements:
- Add a real product inventory database.
- Add real order tracking integration.
- Add escalation to a ticketing system.
- Add human handoff when urgency is high or the customer is dissatisfied.
- Add multilingual support.

## 14. Optional: Local Mock Version Without API Calls

Use this fallback only if you need to demonstrate logic without consuming API credits.
It does not replace the OpenAI implementation, but it can help during offline testing.

In [11]:
def mock_analyze_customer_query(query: str) -> Dict[str, Any]:
    """
    Simple rule-based fallback for offline demonstration.
    """

    lower = query.lower()

    if any(word in lower for word in ["emergency", "urgent", "tomorrow", "urgently"]):
        urgency = "high"
    elif any(word in lower for word in ["problem", "can't", "doesn't work", "receipt", "warranty"]):
        urgency = "medium"
    else:
        urgency = "low"

    if any(word in lower for word in ["thank", "excellent", "great"]):
        sentiment = "positive"
    elif any(word in lower for word in ["can't", "emergency", "never arrived", "doesn't work", "frustrated"]):
        sentiment = "negative"
    else:
        sentiment = "neutral"

    if "receipt" in lower or "visa" in lower or "installments" in lower:
        category = "billing" if "receipt" in lower else "financing"
    elif "return" in lower:
        category = "return"
    elif "warranty" in lower:
        category = "warranty"
    elif "configure" in lower or "install" in lower or "doesn't work" in lower:
        category = "technical"
    elif "recommend" in lower or "stock" in lower:
        category = "product_information"
    else:
        category = "general_information"

    return {
        "customer_sentiment": sentiment,
        "detected_emotions": ["urgency"] if urgency == "high" else [],
        "query_category": category,
        "urgency_level": urgency,
        "mentioned_products": [],
        "extracted_information": {},
        "recommended_routing": f"{category.replace('_', ' ').title()} Team",
        "reasoning_summary": "Rule-based classification for offline demonstration."
    }